# EDA de los datos crudos

Este notebook visualiza los artefactos generados por `scripts/data_eda.py`. El script usa `ffprobe` para obtener duraciones reales, separa los sidecars `._*` de macOS y detecta outliers por dataset con la regla IQR.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = REPO_ROOT / "artifacts" / "data_eda"
SCRIPT = REPO_ROOT / "scripts" / "data_eda.py"

if not (OUTPUT_DIR / "eda_summary.json").exists():
    subprocess.run([sys.executable, str(SCRIPT)], check=True)

OUTPUT_DIR

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
summary = pd.read_csv(OUTPUT_DIR / "dataset_summary.csv")
videos = pd.read_csv(OUTPUT_DIR / "video_metadata.csv")
duration_outliers = pd.read_csv(OUTPUT_DIR / "duration_outliers.csv")
label_outliers = pd.read_csv(OUTPUT_DIR / "label_outliers.csv")
unmatched = pd.read_csv(OUTPUT_DIR / "metadata_unmatched.csv")
payload = json.loads((OUTPUT_DIR / "eda_summary.json").read_text())
summary

## Inventario y duración

In [ ]:
display(pd.DataFrame({
    "metrica": ["archivos", "archivos con extensión de video", "sidecars AppleDouble", "videos reales", "horas de video", "fallos ffprobe"],
    "valor": [payload["file_count"], payload["video_extension_files"], payload["appledouble_video_sidecars"], payload["usable_videos"], round(payload["total_video_seconds"] / 3600, 2), payload["probe_failures"]],
}))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=summary, x="dataset", y="usable_videos", ax=axes[0], color="#3B82F6")
axes[0].set(title="Videos reales por dataset", xlabel="", ylabel="Videos")
sns.barplot(data=summary, x="dataset", y="duration_total_seconds", ax=axes[1], color="#14B8A6")
axes[1].set(title="Duración total por dataset", xlabel="", ylabel="Segundos")
plt.tight_layout()

In [ ]:
valid_videos = videos[(videos["is_appledouble"] == False) & videos["duration_seconds"].notna()].copy()
plt.figure(figsize=(14, 6))
sns.boxplot(data=valid_videos, x="dataset", y="duration_seconds", showfliers=False)
plt.yscale("log")
plt.title("Distribución de duración por dataset (escala log, sin dibujar outliers)")
plt.xlabel("")
plt.ylabel("Duración (segundos, log)")
plt.show()

summary[["dataset", "duration_min_seconds", "duration_median_seconds", "duration_p95_seconds", "duration_max_seconds", "duration_outliers_iqr"]]

## Glosas, frases e idioma

In [ ]:
display(summary[[
    "dataset", "annotation_type", "language", "metadata_rows", "unique_labels",
    "label_words_median", "label_words_p95", "label_words_max",
    "foreign_script_characters"
]])

fig, ax = plt.subplots(figsize=(12, 5))
plot_data = summary.melt(
    id_vars="dataset",
    value_vars=["label_words_median", "label_words_p95"],
    var_name="estadistica",
    value_name="palabras",
)
sns.barplot(data=plot_data, x="dataset", y="palabras", hue="estadistica", ax=ax)
ax.set_yscale("log")
ax.set(title="Longitud de las anotaciones", xlabel="", ylabel="Palabras (log)")
plt.tight_layout()

## Calidad y outliers

In [ ]:
display(summary[[
    "dataset", "metadata_rows", "usable_videos", "exact_id_matches",
    "normalized_id_matches", "probe_failures", "duplicate_video_stems"
]])

print("Outliers de duración:", len(duration_outliers))
display(duration_outliers.sort_values("duration_seconds", ascending=False).head(30))
print("Outliers de longitud textual:", len(label_outliers))
display(label_outliers.sort_values("word_count", ascending=False).head(30))
print("Elementos sin correspondencia normalizada:", len(unmatched))
display(unmatched.head(50))

## Archivos generados

- `REPORT.md`: resumen legible y conclusiones.
- `dataset_summary.csv`: métricas agregadas por dataset.
- `video_metadata.csv`: duración, resolución, FPS, codecs y errores por archivo.
- `duration_outliers.csv`: videos atípicos dentro de cada dataset.
- `label_outliers.csv`: anotaciones atípicas por cantidad de palabras.
- `metadata_unmatched.csv`: videos o IDs sin correspondencia tras normalizar nombres.
- `srt_summary.csv`: resumen de subtítulos/glosas temporizadas.